In [1]:
!pip uninstall -y -q bitsandbytes 2>/dev/null
!rm -rf /usr/local/lib/python3.12/dist-packages/bitsandbytes* 2>/dev/null
!pip install -q transformers==4.57.1 accelerate==1.10.1 peft==0.17.1 scipy scikit-learn matplotlib
print("Install complete")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 52.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.6 MB/s eta 0:00:00
Install complete


In [4]:
import os, sys, json, pickle, time, warnings, random, string, re, gc, types
import importlib.util
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# bitsandbytes stub
def _dummy_spec(name):
    return importlib.util.spec_from_loader(name, loader=None)

class _StubClass:
    def __init__(self, *a, **kw): pass
    def __call__(self, *a, **kw): return None

_bnb_stub = types.ModuleType("bitsandbytes")
_bnb_stub.__version__ = "0.0.0-stub"; _bnb_stub.__path__ = []
_bnb_stub.__spec__ = _dummy_spec("bitsandbytes")

_bnb_nn = types.ModuleType("bitsandbytes.nn")
for n in ["Linear8bitLt","Linear4bit","Int8Params","Params4bit"]:
    setattr(_bnb_nn, n, _StubClass)
_bnb_nn.__spec__ = _dummy_spec("bitsandbytes.nn"); _bnb_stub.nn = _bnb_nn

_bnb_optim = types.ModuleType("bitsandbytes.optim")
for n in ["Adam8bit","AdamW8bit","Adam32bit"]:
    setattr(_bnb_optim, n, _StubClass)
_bnb_optim.__spec__ = _dummy_spec("bitsandbytes.optim"); _bnb_stub.optim = _bnb_optim

_bnb_func = types.ModuleType("bitsandbytes.functional")
_bnb_func.__spec__ = _dummy_spec("bitsandbytes.functional"); _bnb_stub.functional = _bnb_func

_bnb_triton = types.ModuleType("bitsandbytes.triton")
_bnb_triton.__spec__ = _dummy_spec("bitsandbytes.triton"); _bnb_stub.triton = _bnb_triton

for name, mod in [("bitsandbytes",_bnb_stub),("bitsandbytes.nn",_bnb_nn),
                  ("bitsandbytes.optim",_bnb_optim),
                  ("bitsandbytes.functional",_bnb_func),
                  ("bitsandbytes.triton",_bnb_triton)]:
    sys.modules[name] = mod

assert importlib.util.find_spec("bitsandbytes") is not None
print("bitsandbytes stub OK")

import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from collections import Counter
from transformers import (Qwen2_5_VLForConditionalGeneration, AutoProcessor,
                          Owlv2Processor, Owlv2ForObjectDetection,
                          CLIPModel, CLIPProcessor)
from peft import LoraConfig, get_peft_model
warnings.filterwarnings("ignore")

TRAIN_ANN     = "/kaggle/input/datasets/lhanhsin/vizwiz/Annotations/Annotations/train.json"
VAL_ANN       = "/kaggle/input/datasets/lhanhsin/vizwiz/Annotations/Annotations/val.json"
TRAIN_IMG_DIR = "/kaggle/input/datasets/lhanhsin/vizwiz/train/train"
VAL_IMG_DIR   = "/kaggle/input/datasets/lhanhsin/vizwiz/val/val"

QWEN_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
OWL_MODEL_ID  = "google/owlv2-base-patch16-ensemble"
CLIP_MODEL_ID = "openai/clip-vit-base-patch32"

IMAGE_SIZE = 336
PIXEL_BUDGET = IMAGE_SIZE * IMAGE_SIZE

SYSTEM_PROMPT = ("You are a careful visual assistant helping a blind user. "
                 "Answer using only visible information. "
                 "If the answer cannot be determined, respond exactly: unanswerable.")

CACHE_DIR = "/kaggle/working/cache"
LORA_DIR  = "/kaggle/working/qwen_lora_adapter"
os.makedirs(CACHE_DIR, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# v3 sample sizes
N_FINETUNE     = 1800
N_RISK_TRAIN   = 500
N_POLICY_TRAIN = 100
N_CALIBRATION  = 500
N_TEST         = 400
FT_EPOCHS      = 3

print("Config ready")

bitsandbytes stub OK
Config ready


In [5]:
def load_vizwiz(ann_path, img_dir):
    with open(ann_path) as f:
        data = json.load(f)
    return [{"image_id": e["image"],
             "image_path": os.path.join(img_dir, e["image"]),
             "question": e["question"],
             "answers": [a["answer"] for a in e["answers"]]}
            for e in data if os.path.exists(os.path.join(img_dir, e["image"]))]

def answer_type_bucket(s):
    ans = [a.lower().strip() for a in s["answers"]]
    if any(a in ("unanswerable","unsuitable","") for a in ans): return "unanswerable"
    if any(a in ("yes","no") for a in ans): return "yesno"
    if any(a.isdigit() for a in ans): return "number"
    return "other"

all_train = load_vizwiz(TRAIN_ANN, TRAIN_IMG_DIR)
val_pool  = load_vizwiz(VAL_ANN, VAL_IMG_DIR)
rng = random.Random(SEED)
rng.shuffle(all_train)

finetune_samples     = all_train[:N_FINETUNE]
risk_train_samples   = all_train[N_FINETUNE:N_FINETUNE + N_RISK_TRAIN]
policy_train_samples = all_train[N_FINETUNE + N_RISK_TRAIN:
                                  N_FINETUNE + N_RISK_TRAIN + N_POLICY_TRAIN]

buckets = {}
for s in val_pool: buckets.setdefault(answer_type_bucket(s), []).append(s)
full_dist = {k: len(v)/len(val_pool) for k,v in buckets.items()}
val_selected = []
for k, v in buckets.items():
    rng.shuffle(v); val_selected.extend(v[:round(full_dist[k]*(N_CALIBRATION+N_TEST))])
rng.shuffle(val_selected)
calibration_samples = val_selected[:N_CALIBRATION]
test_samples        = val_selected[N_CALIBRATION:N_CALIBRATION+N_TEST]

def ids(s): return set(x["image_id"] for x in s)
assert not (ids(finetune_samples) & ids(risk_train_samples))
assert not (ids(finetune_samples) & ids(policy_train_samples))
assert not (ids(finetune_samples) & ids(test_samples))
assert not (ids(risk_train_samples) & ids(policy_train_samples))
assert not (ids(risk_train_samples) & ids(calibration_samples))
assert not (ids(risk_train_samples) & ids(test_samples))
assert not (ids(policy_train_samples) & ids(calibration_samples))
assert not (ids(policy_train_samples) & ids(test_samples))
assert not (ids(calibration_samples) & ids(test_samples))

print(f"Fine-tune: {len(finetune_samples)}")
print(f"Risk-train: {len(risk_train_samples)}")
print(f"Policy-train: {len(policy_train_samples)}")
print(f"Calibration: {len(calibration_samples)}")
print(f"Test: {len(test_samples)}")

Fine-tune: 1800
Risk-train: 500
Policy-train: 100
Calibration: 500
Test: 400


In [6]:
ARTICLES = {"a","an","the"}; PUNCT = str.maketrans("", "", string.punctuation)
def normalize_answer(ans):
    ans = str(ans).lower().strip().translate(PUNCT)
    return " ".join(w for w in ans.split() if w not in ARTICLES)
def vqa_accuracy(candidate, answers):
    cand = normalize_answer(candidate)
    gt = [normalize_answer(a) for a in answers]
    if len(gt) < 2: return float(cand in gt)
    scores = []
    for i in range(len(gt)):
        others = gt[:i] + gt[i+1:]
        scores.append(min(sum(g == cand for g in others)/3, 1))
    return float(np.mean(scores))
def majority_answer(answers):
    gt = [normalize_answer(a) for a in answers]
    return Counter(gt).most_common(1)[0][0]

assert abs(vqa_accuracy("yes", ["yes","yes","yes"]) - 0.667) < 0.01
assert abs(vqa_accuracy("unsuitable", ["unsuitable"]*10) - 1.0) < 0.01
print("Metrics OK")

Metrics OK


In [7]:
gc.collect(); torch.cuda.empty_cache()
torch.set_default_dtype(torch.float16)

qwen_processor = AutoProcessor.from_pretrained(
    QWEN_MODEL_ID,
    size={"height": IMAGE_SIZE, "width": IMAGE_SIZE},
    min_pixels=PIXEL_BUDGET, max_pixels=PIXEL_BUDGET,
    do_resize=True, resample=Image.BICUBIC)

qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    QWEN_MODEL_ID, dtype=torch.float16, device_map="auto",
    low_cpu_mem_usage=True, max_memory={0:"12GiB",1:"12GiB","cpu":"20GiB"})
qwen_model.eval()
for p in qwen_model.parameters(): p.requires_grad = False

qwen_model.gradient_checkpointing_enable()
qwen_model.enable_input_require_grads()
qwen_model = get_peft_model(qwen_model, LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias="none",
    target_modules=["q_proj","v_proj"], task_type="CAUSAL_LM"))
qwen_model.print_trainable_parameters()

def prepare_image(image):
    return image.convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BICUBIC)

def process_vision_info(messages):
    images, videos = [], []
    for m in messages:
        if m["role"] == "user":
            for c in m["content"]:
                if c["type"] == "image": images.append(c["image"])
                elif c["type"] == "video": videos.append(c["video"])
    return images, videos

def input_device():
    try: return next(qwen_model.parameters()).device
    except: return torch.device("cuda:0")

def build_training_example(sample):
    img = prepare_image(Image.open(sample["image_path"]))
    target = majority_answer(sample["answers"])
    messages = [{"role":"system","content":SYSTEM_PROMPT},
                {"role":"user","content":[{"type":"image","image":img},
                                          {"type":"text","text":sample["question"]}]}]
    prompt = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    images, videos = process_vision_info(messages)
    enc = qwen_processor(text=[prompt], images=images, videos=videos or None, return_tensors="pt")
    target_with_eos = target + qwen_processor.tokenizer.eos_token
    full = qwen_processor(text=[prompt+target_with_eos], images=images, videos=videos or None, return_tensors="pt")
    labels = full["input_ids"].clone()
    labels[:, :enc["input_ids"].shape[1]] = -100
    full["labels"] = labels
    return full

BATCH_SIZE, LR = 4, 2e-4
optimizer = torch.optim.AdamW((p for p in qwen_model.parameters() if p.requires_grad), lr=LR)
qwen_model.train()
skipped = 0
for epoch in range(FT_EPOCHS):
    rng.shuffle(finetune_samples)
    accum, running_loss = 0, 0
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(finetune_samples, desc=f"LoRA {epoch+1}/{FT_EPOCHS}")
    for sample in pbar:
        try:
            enc = build_training_example(sample)
            dev = input_device()
            enc = {k: v.to(dev) if torch.is_tensor(v) else v for k,v in enc.items()}
            out = qwen_model(**enc)
            loss = out.loss / BATCH_SIZE
            if not torch.isfinite(loss):
                del enc, out, loss; optimizer.zero_grad(set_to_none=True); accum=0
                torch.cuda.empty_cache(); continue
            loss.backward()
            running_loss += loss.item(); accum += 1
            if accum == BATCH_SIZE:
                torch.nn.utils.clip_grad_norm_((p for p in qwen_model.parameters() if p.requires_grad), 0.3)
                optimizer.step(); optimizer.zero_grad(set_to_none=True)
                accum=0; pbar.set_postfix(loss=f"{running_loss:.4f}"); running_loss=0
            del enc, out, loss; torch.cuda.empty_cache()
        except torch.cuda.OutOfMemoryError:
            skipped += 1; optimizer.zero_grad(set_to_none=True); accum=0
            gc.collect(); torch.cuda.empty_cache()
        except Exception:
            skipped += 1; optimizer.zero_grad(set_to_none=True); accum=0
            torch.cuda.empty_cache()

qwen_model.eval()
qwen_model.save_pretrained(LORA_DIR); qwen_processor.save_pretrained(LORA_DIR)
print(f"LoRA done. Skipped: {skipped}")

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

LoRA 1/3:   0%|          | 0/1800 [00:00<?, ?it/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


LoRA 2/3:   0%|          | 0/1800 [00:00<?, ?it/s]

LoRA 3/3:   0%|          | 0/1800 [00:00<?, ?it/s]

LoRA done. Skipped: 0


In [8]:
owl_processor = Owlv2Processor.from_pretrained(OWL_MODEL_ID)
owl_model = Owlv2ForObjectDetection.from_pretrained(OWL_MODEL_ID).to("cuda:0").eval()
for p in owl_model.parameters(): p.requires_grad = False
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_ID).to("cuda:0").eval()
for p in clip_model.parameters(): p.requires_grad = False
print("OWLv2 + CLIP loaded")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

OWLv2 + CLIP loaded


In [9]:
SAFETY_KEYWORDS = ["medicine","pill","medication","dosage","prescription","aspirin",
                    "traffic","crosswalk","signal","pedestrian","stop sign",
                    "warning","danger","hazard","poison","toxic","fire","sharp","burn",
                    "expire","expiration","spoiled","moldy","rotten",
                    "allergen","peanut","gluten","dairy","contains",
                    "hospital","doctor","symptom","injury","blood"]

def qwen_answer_deterministic(image, question, max_new_tokens=20):
    prompt_text = f"{SYSTEM_PROMPT}\n\n{question}"
    messages = [{"role":"user","content":[{"type":"image","image":image},
                                            {"type":"text","text":prompt_text}]}]
    text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    images, videos = process_vision_info(messages)
    dev = input_device()
    inputs = qwen_processor(text=[text], images=images, videos=videos or None,
                            return_tensors="pt").to(dev)
    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            return_dict_in_generate=True, output_scores=True,
            do_sample=False, use_cache=True,
            pad_token_id=qwen_processor.tokenizer.pad_token_id,
            eos_token_id=qwen_processor.tokenizer.eos_token_id)
    answer = qwen_processor.decode(outputs.sequences[0, inputs.input_ids.shape[1]:],
                                    skip_special_tokens=True).strip()
    token_probs = [torch.softmax(s, dim=-1).max().item() for s in outputs.scores] if outputs.scores else [0.5]
    del inputs, outputs; torch.cuda.empty_cache()
    return (answer, float(np.mean(token_probs)),
            float(np.var(token_probs)), float(np.min(token_probs)))

def qwen_self_consistency(image, question, n_samples=5, max_new_tokens=15):
    prompt_text = f"{SYSTEM_PROMPT}\n\n{question}"
    messages = [{"role":"user","content":[{"type":"image","image":image},
                                            {"type":"text","text":prompt_text}]}]
    text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    images, videos = process_vision_info(messages)
    dev = input_device()
    inputs = qwen_processor(text=[text], images=images, videos=videos or None,
                            return_tensors="pt").to(dev)
    answers = []
    with torch.no_grad():
        for _ in range(n_samples):
            out = qwen_model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=True, temperature=0.7, top_p=0.9, use_cache=True,
                pad_token_id=qwen_processor.tokenizer.pad_token_id,
                eos_token_id=qwen_processor.tokenizer.eos_token_id)
            answers.append(qwen_processor.decode(out[0, inputs.input_ids.shape[1]:],
                                                  skip_special_tokens=True).strip())
    del inputs; torch.cuda.empty_cache()
    norm_answers = [normalize_answer(a) for a in answers]
    counts = Counter(norm_answers)
    majority_count = counts.most_common(1)[0][1]
    agreement = majority_count / n_samples
    probs = np.array([c / n_samples for c in counts.values()])
    entropy = float(-np.sum(probs * np.log(probs + 1e-8)))
    return agreement, entropy, answers

def qwen_verbalized_confidence(image, question, answer, max_new_tokens=10):
    prompt = (f"Question: {question}\nYour answer: {answer}\n"
              "On a scale of 0 to 100, how confident are you that this answer is correct? "
              "Output only the number.")
    messages = [{"role":"user","content":[{"type":"image","image":image},
                                            {"type":"text","text":prompt}]}]
    text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    images, videos = process_vision_info(messages)
    dev = input_device()
    inputs = qwen_processor(text=[text], images=images, videos=videos or None,
                            return_tensors="pt").to(dev)
    with torch.no_grad():
        outputs = qwen_model.generate(**inputs, max_new_tokens=max_new_tokens,
                                       do_sample=False, use_cache=True,
                                       pad_token_id=qwen_processor.tokenizer.pad_token_id,
                                       eos_token_id=qwen_processor.tokenizer.eos_token_id)
    out = qwen_processor.decode(outputs[0, inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    m = re.search(r"(\d+)", out)
    del inputs, outputs; torch.cuda.empty_cache()
    return float(m.group(1))/100.0 if m else 0.5

def owl_features(image, query_text, threshold=0.1):
    if not query_text or query_text.lower().startswith("unanswerable"): return 0.0, 0
    try:
        query_text = query_text.strip()[:200]
        inputs = owl_processor(images=image, text=[query_text], return_tensors="pt",
                               padding=True, truncation=True, max_length=16).to("cuda:0")
        with torch.no_grad(): outputs = owl_model(**inputs)
        ts = torch.tensor([image.size[::-1]]).to("cuda:0")
        res = owl_processor.post_process_object_detection(outputs, threshold=threshold, target_sizes=ts)
        scores = res[0]["scores"].cpu().numpy()
        del inputs, outputs; torch.cuda.empty_cache()
        return (float(np.max(scores)), int(np.sum(scores < 0.3))) if len(scores) else (0.0, 0)
    except Exception as e:
        print(f"    owl failed: {str(e)[:50]}"); return 0.0, 0

def clip_features(image, answer_text, question_text):
    if not answer_text or answer_text.lower().startswith("unanswerable"): return 0.0, 0.0
    try:
        answer_text = answer_text.replace("\n"," ").strip()[:100]
        question_text = question_text.replace("\n"," ").strip()[:100]
        words = answer_text.split()
        fragments = [" ".join(words[:i+1]) for i in range(len(words))] or [answer_text]
        texts = [f"{question_text} {f}"[:200] for f in fragments]
        inputs = clip_processor(text=texts, images=image, return_tensors="pt",
                                 padding=True, truncation=True, max_length=77).to("cuda:0")
        with torch.no_grad():
            out = clip_model(**inputs)
            img_e = out.image_embeds / out.image_embeds.norm(dim=-1, keepdim=True)
            txt_e = out.text_embeds / out.text_embeds.norm(dim=-1, keepdim=True)
            sims = (img_e @ txt_e.T).squeeze(0).cpu().numpy()
        del inputs, out; torch.cuda.empty_cache()
        return float(sims[-1]), float(np.var(sims))
    except Exception as e:
        print(f"    clip failed: {str(e)[:50]}"); return 0.0, 0.0

def question_features(question):
    q = question.lower().strip()
    has_safety_kw = int(any(kw in q for kw in SAFETY_KEYWORDS))
    is_yesno = int(any(q.startswith(w) for w in
        ["is ","are ","was ","were ","does ","do ","did ","can ","could "]))
    return has_safety_kw, is_yesno

print("Feature functions v3 ready")

Feature functions v3 ready


In [10]:
def build_cache(samples, split_name, cache_file):
    if os.path.exists(cache_file):
        with open(cache_file, 'rb') as f: return pickle.load(f)

    cache, errors = [], []
    pbar = tqdm(samples, desc=f"[{split_name}]")
    for idx, sample in enumerate(pbar):
        try:
            img = Image.open(sample["image_path"]).convert("RGB")
            q, answers = sample["question"], sample["answers"]

            answer, prob_mean, prob_var, prob_min = qwen_answer_deterministic(img, q)
            owl_conf, owl_low = owl_features(img, answer)
            clip_sim, clip_var = clip_features(img, answer, q)
            verbalized = qwen_verbalized_confidence(img, q, answer)
            sc_agreement, sc_entropy, _ = qwen_self_consistency(img, q, n_samples=5)
            has_safety_kw, is_yesno = question_features(q)
            acc = vqa_accuracy(answer, answers)

            # 11-dim state
            state = np.array([owl_conf, owl_low, clip_sim, clip_var,
                               prob_mean, prob_var, prob_min,
                               sc_agreement, sc_entropy,
                               has_safety_kw, is_yesno], dtype=np.float32)

            cache.append({
                "image_id": sample["image_id"], "question": q, "answers": answers,
                "candidate_answer": answer, "base_accuracy": acc,
                "state_raw": state,
                "verbalized_confidence": verbalized,
                "unsafe_proxy_label": 0,   # relabel later
                "owl_conf": owl_conf, "clip_sim": clip_sim,
                "sc_agreement": sc_agreement, "sc_entropy": sc_entropy,
                "has_safety_kw": has_safety_kw, "is_yesno": is_yesno,
            })
        except Exception as e:
            errors.append((idx, sample.get("image_id"), type(e).__name__, str(e)[:80]))
            continue
        if len(cache) % 50 == 0:
            torch.cuda.empty_cache()
            pbar.set_postfix({"ok": len(cache), "err": len(errors)})

    if errors:
        print(f"\n{len(errors)}/{len(samples)} failed in {split_name}:")
        for e in errors[:5]: print(f"  {e}")

    with open(cache_file, "wb") as f: pickle.dump(cache, f)
    print(f"{split_name}: {len(cache)}/{len(samples)} cached")
    return cache

!rm -f /kaggle/working/cache/*.pkl

risk_train_raw   = build_cache(risk_train_samples,   "risk_train",   f"{CACHE_DIR}/risk_train_raw_cache.pkl")
policy_train_raw = build_cache(policy_train_samples, "policy_train", f"{CACHE_DIR}/policy_train_raw_cache.pkl")
calibration_raw  = build_cache(calibration_samples,  "calibration",  f"{CACHE_DIR}/calibration_raw_cache.pkl")
test_raw         = build_cache(test_samples,         "test",         f"{CACHE_DIR}/test_raw_cache.pkl")

[risk_train]:   0%|          | 0/500 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


risk_train: 500/500 cached


[policy_train]:   0%|          | 0/100 [00:00<?, ?it/s]

policy_train: 100/100 cached


[calibration]:   0%|          | 0/500 [00:00<?, ?it/s]

calibration: 500/500 cached


[test]:   0%|          | 0/400 [00:00<?, ?it/s]

test: 400/400 cached


In [11]:
def safety_proxy_label_v2(candidate_answer, base_accuracy):
    """Confidently-wrong proxy: incorrect answer that isn't a hedge."""
    if base_accuracy >= 0.5: return 0
    ans = str(candidate_answer).strip().lower()
    if ans in ("unanswerable","unsuitable","") or \
       ans.startswith("unanswerable") or ans.startswith("unsuitable"):
        return 0
    return 1

for name, cache in [("risk_train", risk_train_raw),
                     ("policy_train", policy_train_raw),
                     ("calibration", calibration_raw),
                     ("test", test_raw)]:
    for e in cache:
        e["unsafe_proxy_label"] = safety_proxy_label_v2(e["candidate_answer"], e["base_accuracy"])
    pos = sum(e["unsafe_proxy_label"] for e in cache)
    print(f"  {name:16s}: {pos}/{len(cache)} positives ({100*pos/len(cache):.1f}%)")
    with open(f"{CACHE_DIR}/{name}_raw_cache.pkl", "wb") as f:
        pickle.dump(cache, f)

!rm -rf /kaggle/working/aerpn_export_v3
!mkdir -p /kaggle/working/aerpn_export_v3
!cp /kaggle/working/cache/*_raw_cache.pkl /kaggle/working/aerpn_export_v3/
!cp -r /kaggle/working/qwen_lora_adapter /kaggle/working/aerpn_export_v3/

meta = {"version": "v3", "state_dim": 11,
        "n_risk_train": len(risk_train_raw),
        "n_policy_train": len(policy_train_raw),
        "n_calibration": len(calibration_raw),
        "n_test": len(test_raw),
        "proxy_definition": "confidently_wrong_v2"}
with open("/kaggle/working/aerpn_export_v3/metadata.json", "w") as f:
    json.dump(meta, f, indent=2)
print(json.dumps(meta, indent=2))

!cd /kaggle/working && zip -r aerpn_export_v3.zip aerpn_export_v3/
print("\nDownload aerpn_export_v3.zip → upload as Kaggle dataset 'aerpn-cache-v3'")

  risk_train      : 123/500 positives (24.6%)
  policy_train    : 26/100 positives (26.0%)
  calibration     : 116/500 positives (23.2%)
  test            : 110/400 positives (27.5%)
{
  "version": "v3",
  "state_dim": 11,
  "n_risk_train": 500,
  "n_policy_train": 100,
  "n_calibration": 500,
  "n_test": 400,
  "proxy_definition": "confidently_wrong_v2"
}
  adding: aerpn_export_v3/ (stored 0%)
  adding: aerpn_export_v3/calibration_raw_cache.pkl (deflated 77%)
  adding: aerpn_export_v3/policy_train_raw_cache.pkl (deflated 74%)
  adding: aerpn_export_v3/test_raw_cache.pkl (deflated 76%)
  adding: aerpn_export_v3/risk_train_raw_cache.pkl (deflated 76%)
  adding: aerpn_export_v3/qwen_lora_adapter/ (stored 0%)
  adding: aerpn_export_v3/qwen_lora_adapter/preprocessor_config.json (deflated 52%)
  adding: aerpn_export_v3/qwen_lora_adapter/merges.txt (deflated 57%)
  adding: aerpn_export_v3/qwen_lora_adapter/tokenizer_config.json (deflated 88%)
  adding: aerpn_export_v3/qwen_lora_adapter/READM